<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/EarningsLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade yfinance
!pip install scipy==1.16.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: yfinance
    Found existing installation: yfinance 0.2.66
    Uninstalling yfinance-0.2.66:
      Successfully uninstalled yfinance-0.2.66
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 38.1 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from math import sqrt
print("Libraries installed successfully!")

Libraries installed successfully!


## Build earnings lab class

In [16]:

# =============================================================
# HELPERS
# =============================================================

def find_next_trading_day(price_data, target_date, max_days=10):
    trading_days = sorted(price_data.index)
    for d in trading_days:
        if d >= target_date:
            return d
    return None


def get_bias_and_structure(ratio, iv_rank):
    """
    Returns (bias, suggested_structure) given a ratio and iv_rank.
    Extracted as a standalone function so it can be called for
    both the median-based and mean-based ratios independently.
    """

    if ratio > 1.5 and (iv_rank is None or iv_rank > 60):
        return "STRONG SELL VOLATILITY", "Short Iron Condor"

    elif ratio > 1.3:
        return "SELL VOLATILITY", "Short Iron Condor"

    elif ratio < 0.6 and (iv_rank is None or iv_rank < 40):
        return "STRONG BUY VOLATILITY", "Long Straddle / Strangle"

    elif ratio < 0.80:
        return "BUY VOLATILITY", "Long Straddle"

    else:
        return "NEUTRAL / FAIR", "Directional or Calendar Spread"


def consistency_label(structure_median, structure_mean):
    """
    Returns 'Same' if both approaches suggest the same structure,
    'Different' otherwise.
    """
    return "Same" if structure_median == structure_mean else "Different"


# =============================================================
# SINGLE TICKER ENGINE
# =============================================================

def earnings_edge_engine(ticker_symbol, lookback=24):
    """
    Analyze a single ticker's earnings edge.

    Parameters
    ----------
    ticker_symbol : str
    lookback      : int
        Number of past earnings reports to include.
        Default raised to 24 (~6 years of quarterly reports)
        for a more statistically robust sample.
        Set higher (e.g. 40) for large caps with long history.
    """

    print(f"\nAnalyzing {ticker_symbol}...")

    ticker = yf.Ticker(ticker_symbol)

    # =====================================================
    # 1. EARNINGS DATES
    # =====================================================
    earnings = ticker.get_earnings_dates(limit=lookback)

    if earnings is None or earnings.empty:
        print(f"No earnings data for {ticker_symbol}")
        return None

    earnings = earnings.reset_index()
    earnings["Earnings Date"] = pd.to_datetime(
        earnings["Earnings Date"]
    ).dt.tz_localize(None)

    today = pd.Timestamp.today().normalize()

    past_earnings   = earnings[earnings["Earnings Date"] < today]
    future_earnings = earnings[earnings["Earnings Date"] >= today]

    if future_earnings.empty:
        print(f"No upcoming earnings for {ticker_symbol}")
        return None

    next_earnings_date = future_earnings.iloc[0]["Earnings Date"].date()

    # how many past reports are we actually using
    reports_used = len(past_earnings)
    print(f"  Using {reports_used} past earnings reports")

    # =====================================================
    # 2. PRICE DATA & HISTORICAL MOVES
    # =====================================================

    # extend to 7y to support larger lookback windows
    price_data = ticker.history(period="7y", auto_adjust=True)
    price_data.index = price_data.index.date

    results = []

    for _, row in past_earnings.iterrows():

        announce_date = row["Earnings Date"].date()
        event_day     = find_next_trading_day(price_data, announce_date)

        if not event_day:
            continue

        prior_day = find_next_trading_day(
            price_data, announce_date - timedelta(days=7)
        )

        if not prior_day or prior_day >= event_day:
            continue

        try:
            prior_close    = price_data.loc[prior_day]["Close"]
            next_day       = find_next_trading_day(
                price_data, event_day + timedelta(days=1)
            )

            if not next_day:
                continue

            next_close     = price_data.loc[next_day]["Close"]
            two_day_pct    = (next_close - prior_close) / prior_close * 100
            raw_move       = two_day_pct  # signed — preserve direction

            results.append({
                "Earnings Date":       announce_date,
                "Event Day":           event_day,
                "Post Earnings Move %": round(abs(two_day_pct), 2),
                "Signed Move %":       round(raw_move, 2),
                "Direction":           "Up" if two_day_pct > 0 else "Down",
            })

        except Exception:
            continue

    if not results:
        print(f"No valid historical moves for {ticker_symbol}")
        return None

    df_moves = pd.DataFrame(results)

    n_moves = len(df_moves)

    # ── Core statistics
    hist_median   = df_moves["Post Earnings Move %"].median()
    hist_mean     = df_moves["Post Earnings Move %"].mean()
    hist_std      = df_moves["Post Earnings Move %"].std()
    hist_min      = df_moves["Post Earnings Move %"].min()
    hist_max      = df_moves["Post Earnings Move %"].max()

    # ── Signed stats (direction-aware)
    signed_mean   = df_moves["Signed Move %"].mean()
    signed_median = df_moves["Signed Move %"].median()

    # ── Win/loss counts
    up_moves   = df_moves[df_moves["Direction"] == "Up"].shape[0]
    down_moves = df_moves[df_moves["Direction"] == "Down"].shape[0]
    up_pct     = round(up_moves / n_moves * 100, 1)

    directional_bias = "Bullish" if up_moves > down_moves else "Bearish"

    # ── Top 3 largest moves
    top3 = df_moves.nlargest(3, "Post Earnings Move %")[
        ["Earnings Date", "Post Earnings Move %", "Direction"]
    ]

    # =====================================================
    # 3. OPTIONS + IV RANK
    # =====================================================
    expirations = ticker.options

    if not expirations:
        print(f"No options for {ticker_symbol}")
        return None

    exp_dates  = [pd.to_datetime(e).date() for e in expirations]
    valid_exps = [e for e in exp_dates if e > next_earnings_date]

    if not valid_exps:
        print(f"No valid expiration after earnings for {ticker_symbol}")
        return None

    target_exp   = min(valid_exps)
    option_chain = ticker.option_chain(str(target_exp))

    calls         = option_chain.calls
    puts          = option_chain.puts
    current_price = price_data.iloc[-1]["Close"]

    calls["distance"] = abs(calls["strike"] - current_price)
    puts["distance"]  = abs(puts["strike"]  - current_price)

    atm_call = calls.loc[calls["distance"].idxmin()]
    atm_put  = puts.loc[puts["distance"].idxmin()]

    call_mid       = (atm_call["bid"] + atm_call["ask"]) / 2
    put_mid        = (atm_put["bid"]  + atm_put["ask"])  / 2
    straddle_price = call_mid + put_mid

    implied_move_pct = (straddle_price / current_price) * 100

    current_iv = (
        atm_call["impliedVolatility"] + atm_put["impliedVolatility"]
    ) / 2 * 100

    try:
        hist_vol = (
            ticker.history(period="1y")["Close"]
            .pct_change()
            .std()
            * np.sqrt(252)
            * 100
        )
        iv_rank = min(
            max((current_iv - 20) / (hist_vol * 1.5), 0), 100
        )
    except Exception:
        iv_rank = None

    # =====================================================
    # 4. DUAL BIAS LOGIC — MEDIAN AND MEAN
    # =====================================================

    ratio_median = (
        implied_move_pct / hist_median if hist_median > 0 else 1.0
    )
    ratio_mean = (
        implied_move_pct / hist_mean if hist_mean > 0 else 1.0
    )

    # bias and structure based on median
    bias_median, structure_median = get_bias_and_structure(
        ratio_median, iv_rank
    )

    # bias and structure based on mean
    bias_mean, structure_mean = get_bias_and_structure(
        ratio_mean, iv_rank
    )

    # consistency check
    recommendation_consistency = consistency_label(
        structure_median, structure_mean
    )

    # bias scores
    bias_score_median = round((ratio_median - 1) * 100, 2)
    bias_score_mean   = round((ratio_mean   - 1) * 100, 2)

    # =====================================================
    # 5. PRINT OUTPUT
    # =====================================================

    print(f"\n{'='*55}")
    print(f"  {ticker_symbol} — Earnings Edge Analysis")
    print(f"{'='*55}")
    print(f"  Next Earnings Date   : {next_earnings_date}")
    print(f"  Reports Used         : {n_moves} past quarters")
    print(f"  Current Price        : ${round(current_price, 2)}")
    print(f"  Implied Move         : {round(implied_move_pct, 2)}%")
    print(f"  Straddle Price       : ${round(straddle_price, 2)}")
    if iv_rank is not None:
        print(f"  IV Rank              : {round(iv_rank, 1)}%")
    print()
    print(f"  ── Historical Move Stats ({n_moves} reports) ──")
    print(f"  Median Move          : {round(hist_median, 2)}%")
    print(f"  Mean Move            : {round(hist_mean, 2)}%")
    print(f"  Std Dev              : {round(hist_std, 2)}%")
    print(f"  Min / Max            : {round(hist_min, 2)}% / {round(hist_max, 2)}%")
    print(f"  Up / Down            : {up_moves} Up ({up_pct}%) / {down_moves} Down")
    print(f"  Directional Bias     : {directional_bias}")
    print(f"  Signed Mean Move     : {round(signed_mean, 2)}%")
    print(f"  Signed Median Move   : {round(signed_median, 2)}%")
    print()
    print(f"  ── Ratio Analysis ──")
    print(f"  Ratio (Imp/Median)   : {round(ratio_median, 2)}x  → {bias_median}")
    print(f"  Ratio (Imp/Mean)     : {round(ratio_mean, 2)}x  → {bias_mean}")
    print(f"  Structure (Median)   : {structure_median}")
    print(f"  Structure (Mean)     : {structure_mean}")
    print(f"  Recommendation       : {recommendation_consistency}")
    print()
    print(f"  ── Top 3 Largest Historical Moves ──")
    print(top3.to_string(index=False))
    print()

    # =====================================================
    # 6. SUMMARY DATAFRAME
    # =====================================================

    summary_df = pd.DataFrame([{
        "Ticker":                  ticker_symbol,
        "Next Earnings":           next_earnings_date,
        "Reports Used":            n_moves,
        "Current Price":           round(current_price, 2),
        "Implied Move %":          round(implied_move_pct, 2),
        "Hist Median %":           round(hist_median, 2),
        "Hist Mean %":             round(hist_mean, 2),
        "Hist Std %":              round(hist_std, 2),
        "Hist Min %":              round(hist_min, 2),
        "Hist Max %":              round(hist_max, 2),
        "Up Count":                up_moves,
        "Down Count":              down_moves,
        "Up %":                    up_pct,
        "Directional Bias":        directional_bias,
        "Signed Mean %":           round(signed_mean, 2),
        "Signed Median %":         round(signed_median, 2),
        "Ratio (Median)":          round(ratio_median, 2),
        "Ratio (Mean)":            round(ratio_mean, 2),
        "IV Rank %":               round(iv_rank, 1) if iv_rank is not None else None,
        "Bias Score (Median)":     bias_score_median,
        "Bias Score (Mean)":       bias_score_mean,
        "Bias (Median)":           bias_median,
        "Bias (Mean)":             bias_mean,
        "Structure (Median)":      structure_median,
        "Structure (Mean)":        structure_mean,
        "Recommendation":          recommendation_consistency,
        "Top 3 Moves":             top3.to_dict(orient="records"),
    }])

    return summary_df


# =============================================================
# BATCH ENGINE
# =============================================================

def earnings_edge_batch(tickers, lookback=24):
    """
    Run earnings edge analysis on a list of tickers.

    Parameters
    ----------
    tickers  : list or str
    lookback : int
        Number of past earnings to fetch per ticker.
        Default 24 = ~6 years of quarterly data.
        Increase to 40 for the deepest available history
        on large caps (AAPL, MSFT etc go back further).
    """

    if isinstance(tickers, str):
        tickers = [tickers]

    all_results = []

    for t in tickers:
        try:
            result = earnings_edge_engine(t, lookback=lookback)
            if result is not None:
                all_results.append(result)
        except Exception as e:
            print(f"Error on {t}: {e}")

    if not all_results:
        print("No results returned.")
        return pd.DataFrame()

    df = pd.concat(all_results, ignore_index=True)

    # ── Summary table
    display_cols = [
        "Ticker",
        "Next Earnings",
        "Reports Used",
        "Implied Move %",
        "Hist Median %",
        "Hist Mean %",
        "Ratio (Median)",
        "Ratio (Mean)",
        "IV Rank %",
        "Bias (Median)",
        "Bias (Mean)",
        "Structure (Median)",
        "Structure (Mean)",
        "Recommendation",       # Same / Different
        "Directional Bias",
        "Up %",
    ]

    print(f"\n{'='*55}")
    print(f"  FINAL SUMMARY TABLE — {len(df)} tickers")
    print(f"{'='*55}\n")

    print(
        df[display_cols]
        .sort_values("Ratio (Median)", ascending=False)
        .to_string(index=False)
    )

    # ── Flag conflicts — where median and mean disagree
    conflicts = df[df["Recommendation"] == "Different"]

    if not conflicts.empty:
        print(f"\n{'='*55}")
        print(f"  ⚠️  CONFLICTING SIGNALS — {len(conflicts)} ticker(s)")
        print(f"  Mean and Median suggest different structures.")
        print(f"  Use IV Rank and Directional Bias to decide.")
        print(f"{'='*55}")
        print(
            conflicts[[
                "Ticker", "Structure (Median)", "Structure (Mean)",
                "IV Rank %", "Directional Bias"
            ]].to_string(index=False)
        )

    return df



In [17]:
# =============================================================
# EXAMPLE USAGE
# =============================================================

if __name__ == "__main__":

    tickers = ["NVDA", "AAPL", "META", "MSFT", "AMZN"]

    # ── Single ticker deep dive
    result = earnings_edge_engine("NVDA", lookback=32)

    # ── Batch with extended lookback
    df = earnings_edge_batch(tickers, lookback=32)

df


Analyzing NVDA...
  Using 49 past earnings reports

  NVDA — Earnings Edge Analysis
  Next Earnings Date   : 2026-05-20
  Reports Used         : 27 past quarters
  Current Price        : $225.32
  Implied Move         : 7.53%
  Straddle Price       : $16.98
  IV Rank              : 1.1%

  ── Historical Move Stats (27 reports) ──
  Median Move          : 6.28%
  Mean Move            : 7.01%
  Std Dev              : 5.78%
  Min / Max            : 0.17% / 25.85%
  Up / Down            : 18 Up (66.7%) / 9 Down
  Directional Bias     : Bullish
  Signed Mean Move     : 2.89%
  Signed Median Move   : 2.72%

  ── Ratio Analysis ──
  Ratio (Imp/Median)   : 1.2x  → NEUTRAL / FAIR
  Ratio (Imp/Mean)     : 1.07x  → NEUTRAL / FAIR
  Structure (Median)   : Directional or Calendar Spread
  Structure (Mean)     : Directional or Calendar Spread
  Recommendation       : Same

  ── Top 3 Largest Historical Moves ──
Earnings Date  Post Earnings Move % Direction
   2023-05-24                 25.85       

,Ticker,Next Earnings,Reports Used,Current Price,Implied Move %,Hist Median %,Hist Mean %,Hist Std %,Hist Min %,Hist Max %,...,Ratio (Mean),IV Rank %,Bias Score (Median),Bias Score (Mean),Bias (Median),Bias (Mean),Structure (Median),Structure (Mean),Recommendation,Top 3 Moves
0,NVDA,2026-05-20,27,225.32,7.53,6.28,7.01,5.78,0.17,25.85,...,1.07,1.1,19.96,7.40,NEUTRAL / FAIR,NEUTRAL / FAIR,Directional or Calendar Spread,Directional or Calendar Spread,Same,"[{'Earnings Date': 2023-05-24, 'Post Earnings ..."
1,AAPL,2026-07-30,28,300.23,10.54,4.22,4.33,2.99,0.01,14.45,...,2.44,0.2,149.81,143.58,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2020-07-30, 'Post Earnings ..."
2,META,2026-07-29,28,614.23,15.34,9.10,9.92,7.92,0.54,33.41,...,1.55,0.3,68.71,54.70,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2023-02-01, 'Post Earnings ..."
3,MSFT,2026-07-29,28,421.92,13.28,2.83,3.59,2.93,0.11,13.62,...,3.70,0.4,370.25,270.01,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2025-04-30, 'Post Earnings ..."
4,AMZN,2026-07-30,28,264.14,14.05,5.38,5.87,4.23,0.30,16.19,...,2.39,0.3,161.31,139.47,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2022-04-28, 'Post Earnings ..."


In [18]:
# Sort by richest volatility (best for selling premium)
df.sort_values(by="Ratio (Mean)", ascending=False)

,Ticker,Next Earnings,Reports Used,Current Price,Implied Move %,Hist Median %,Hist Mean %,Hist Std %,Hist Min %,Hist Max %,...,Ratio (Mean),IV Rank %,Bias Score (Median),Bias Score (Mean),Bias (Median),Bias (Mean),Structure (Median),Structure (Mean),Recommendation,Top 3 Moves
3,MSFT,2026-07-29,28,421.92,13.28,2.83,3.59,2.93,0.11,13.62,...,3.70,0.4,370.25,270.01,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2025-04-30, 'Post Earnings ..."
1,AAPL,2026-07-30,28,300.23,10.54,4.22,4.33,2.99,0.01,14.45,...,2.44,0.2,149.81,143.58,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2020-07-30, 'Post Earnings ..."
4,AMZN,2026-07-30,28,264.14,14.05,5.38,5.87,4.23,0.30,16.19,...,2.39,0.3,161.31,139.47,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2022-04-28, 'Post Earnings ..."
2,META,2026-07-29,28,614.23,15.34,9.10,9.92,7.92,0.54,33.41,...,1.55,0.3,68.71,54.70,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2023-02-01, 'Post Earnings ..."
0,NVDA,2026-05-20,27,225.32,7.53,6.28,7.01,5.78,0.17,25.85,...,1.07,1.1,19.96,7.40,NEUTRAL / FAIR,NEUTRAL / FAIR,Directional or Calendar Spread,Directional or Calendar Spread,Same,"[{'Earnings Date': 2023-05-24, 'Post Earnings ..."


In [19]:
# Or see only the ones worth trading
df[df["Ratio (Mean)"] > 1.25]   # Good for selling vol

,Ticker,Next Earnings,Reports Used,Current Price,Implied Move %,Hist Median %,Hist Mean %,Hist Std %,Hist Min %,Hist Max %,...,Ratio (Mean),IV Rank %,Bias Score (Median),Bias Score (Mean),Bias (Median),Bias (Mean),Structure (Median),Structure (Mean),Recommendation,Top 3 Moves
1,AAPL,2026-07-30,28,300.23,10.54,4.22,4.33,2.99,0.01,14.45,...,2.44,0.2,149.81,143.58,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2020-07-30, 'Post Earnings ..."
2,META,2026-07-29,28,614.23,15.34,9.10,9.92,7.92,0.54,33.41,...,1.55,0.3,68.71,54.70,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2023-02-01, 'Post Earnings ..."
3,MSFT,2026-07-29,28,421.92,13.28,2.83,3.59,2.93,0.11,13.62,...,3.70,0.4,370.25,270.01,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2025-04-30, 'Post Earnings ..."
4,AMZN,2026-07-30,28,264.14,14.05,5.38,5.87,4.23,0.30,16.19,...,2.39,0.3,161.31,139.47,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2022-04-28, 'Post Earnings ..."
